# Semana 4: duas sementes do codificador congelado contra o checkpoint da IBM, só na validação

Item da semana 4 do doc 20b. Estima as duas entradas do efeito mínimo detectável do doc 22, seção 14.4: quanto o AUROC de detecção de erro de dois modelos varia junto quando os mesmos chips são reamostrados, e quanto ele varia entre sementes de um regime. Nenhuma hipótese é testada aqui e nenhuma escolha de receita depende do resultado. A regra foi fixada antes de rodar, em `prereg/piloto_sementes.md` (commit `8e90879`).

| Modelo | O que é | Roda aqui? |
|---|---|---|
| R1, sementes 101 e 102 | Codificador congelado, lr 5e-4 (grade da semana 3), 50 épocas, receita idêntica à da grade. Sementes só do piloto: as confirmatórias continuam 1 a 5 | Sim, cerca de 10,5 min de treino cada na T4 |
| R0 | Checkpoint ajustado da IBM | Não: usa os escores da val salvos em 2026-06-07, os mesmos da semana 1 |

**Só a validação é lida.** Teste, Bolívia e WorldFloods v2 não são tocados. Abra o Colab logada na conta do mestrado, com GPU T4. Tempo total: cerca de 25 a 30 minutos.

1. Instalação com versões exatas. Se a célula pedir, reinicie a sessão (Ambiente de execução > Reiniciar sessão) e continue da célula 2.
2. Drive, pastas e escores do R0.
3. Backbone pré-treinado, conferido pelo hash.
4. Chips de treino e de val no disco local, com as listas conferidas contra o repositório.
5. Treino das duas sementes do R1 e inferência em 32 bits na val. Se a sessão cair, rode de novo a partir da célula 2: as sementes já feitas são puladas.
6. Análise e registro em `resultados/semana4/piloto_sementes.json`.

Guarde a saída das células 5 e 6.

In [ ]:
!pip install -q "terratorch[peft]==1.2.11" "torchgeo==0.7.1" "peft==0.21.2"
!pip uninstall -y -q torchao

from importlib.metadata import version
for pacote in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy'):
    print(f'{pacote:11s} {version(pacote)}')
assert version('terratorch') == '1.2.11', 'Versão do TerraTorch diferente da fixada.'

import sys
if 'numpy' in sys.modules and sys.modules['numpy'].__version__ != version('numpy'):
    print(f"\nREINICIE A SESSÃO: o NumPy carregado é o {sys.modules['numpy'].__version__} e o instalado é o {version('numpy')}.")
    print('Menu Ambiente de execução > Reiniciar sessão (não precisa excluir o ambiente) e rode a partir da célula 2.')
else:
    print('\nNumPy carregado e instalado coincidem. Pode seguir para a célula 2.')

In [ ]:
import os, re, json, time, hashlib, datetime, shutil
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

RES = f'{RAIZ}/resultados/semana4'
os.makedirs(RES, exist_ok=True)

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

CHAVE_RX = re.compile(r'([A-Za-z-]+_\d+)')
def chave_de(nome):
    achado = CHAVE_RX.search(os.path.basename(str(nome)))
    assert achado, f'Nome de chip sem chave reconhecível: {nome!r}'
    return achado.group(1)

ARQUIVOS_R0 = ('cal_scores.npy', 'cal_labels.npy', 'cal_chip_sizes.npy', 'cal_chip_names.npy')
PULAR = {'S1Hand', 'S2Hand', 'LabelHand', 'S1OtsuLabelHand', 'JRCWaterHand', 'S1Weak', 'S2IndexLabelWeak',
         'S1OtsuLabelWeak', 'venv', '.venv', 'site-packages', '__pycache__'}

def procurar(raiz, nomes):
    achados = {}
    for pasta, subpastas, arquivos in os.walk(raiz, followlinks=True):
        subpastas[:] = [d for d in subpastas if not d.startswith('.') and d not in PULAR]
        for f in arquivos:
            if f in nomes:
                achados.setdefault(f, []).append(os.path.join(pasta, f))
    return achados

t0 = time.time()
ACHADOS_R0 = procurar(RAIZ, ARQUIVOS_R0)
if len(ACHADOS_R0) < len(ARQUIVOS_R0):
    ACHADOS_R0 = procurar(MEUDRIVE, ARQUIVOS_R0)
faltam = [n for n in ARQUIVOS_R0 if n not in ACHADOS_R0]
assert not faltam, (f'Não achei {faltam}. São os escores do R0 usados na semana 1; confira se os atalhos das pastas '
                    'paper_rs2024_* estão em Mestrado/dados.')
for nome in ARQUIVOS_R0:
    if len(ACHADOS_R0[nome]) > 1:
        print(f'Aviso: {nome} aparece {len(ACHADOS_R0[nome])} vezes; uso o primeiro.')
    print(f'{nome:22s} {ACHADOS_R0[nome][0]}')
CAL = {nome: ACHADOS_R0[nome][0] for nome in ARQUIVOS_R0}
print(f'Busca em {time.time() - t0:.0f} s. Resultados em {RES}')

In [ ]:
from huggingface_hub import hf_hub_download

REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
ARQUIVO = 'Prithvi_EO_V2_300M_TL.pt'
REVISAO = '63adbd39c271da4c42f447e69b1a7c91a338cdc9'
SHA_BACKBONE = '3629cedfbb350faafcb0dac902ae0d3c927e25ce8d9e0024aa1276ec66956ddb'

no_drive = f'{RAIZ}/modelo_fixado/backbone/{ARQUIVO}'
if not os.path.exists(no_drive):
    print('Backbone ausente no Drive; baixando na revisão fixada.')
    hf_hub_download(REPO, ARQUIVO, revision=REVISAO, local_dir=os.path.dirname(no_drive))
BACKBONE_PT = f'/content/{ARQUIVO}'
if not (os.path.exists(BACKBONE_PT) and sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE):
    shutil.copy(no_drive, BACKBONE_PT)
assert sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE, 'sha256 do backbone diferente do registrado em modelo.md.'
print(f'Backbone conferido ({SHA_BACKBONE[:8]}…), cópia local em {BACKBONE_PT}')

In [ ]:
import urllib.request, concurrent.futures as cf

BUCKET = 'https://storage.googleapis.com/sen1floods11/v1.1/'
DADOS = '/content/sen1floods11'
S2 = f'{DADOS}/v1.1/data/flood_events/HandLabeled/S2Hand'
LB = f'{DADOS}/v1.1/data/flood_events/HandLabeled/LabelHand'
SP = f'{DADOS}/v1.1/splits/flood_handlabeled'
SPLITS_DRIVE = f'{RAIZ}/dados/splits'
for d in (S2, LB, SP, SPLITS_DRIVE):
    os.makedirs(d, exist_ok=True)

SHA_LISTA = {'treino': '4d77d923fcedcf1dca02606ec15acfd2e90238ea06f0b40169b470ca4bd756b5',
             'val': '14a372859fb18b1f7f9791810b7a233c2cd53aaab84f74721f9ac3476ba50504'}

def chaves_csv(nome):
    destino = os.path.join(SPLITS_DRIVE, nome)
    if not os.path.exists(destino):
        urllib.request.urlretrieve(BUCKET + 'splits/flood_handlabeled/' + nome, destino)
    chaves = []
    for linha in open(destino).read().strip().splitlines():
        primeiro = os.path.basename(linha.split(',')[0].strip())
        chave = re.sub(r'_(S1Hand|S2Hand|LabelHand)(\.tif)?$', '', primeiro)
        assert chave and chave != primeiro, f'Linha inesperada em {nome}: {linha!r}'
        chaves.append(chave)
    return chaves

TREINO, VAL = chaves_csv('flood_train_data.csv'), chaves_csv('flood_valid_data.csv')
for papel, chaves in (('treino', TREINO), ('val', VAL)):
    lista = '\n'.join(sorted(f'{c}_LabelHand.tif' for c in chaves)) + '\n'
    assert hashlib.sha256(lista.encode()).hexdigest() == SHA_LISTA[papel], (
        f'Lista de {papel} diferente da registrada em splits/README.md.')
assert not set(TREINO) & set(VAL)
VAL_SET = set(VAL)

def baixar(url_e_destino):
    url, destino = url_e_destino
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        return 0
    for tentativa in range(4):
        try:
            urllib.request.urlretrieve(url, destino + '.parcial')
            os.replace(destino + '.parcial', destino)
            return 1
        except Exception as erro:
            ultimo = erro
            time.sleep(2 * (tentativa + 1))
    raise RuntimeError(f'Falhou: {url} ({ultimo})')

trabalhos = []
for chave in TREINO + VAL:
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/S2Hand/{chave}_S2Hand.tif', f'{S2}/{chave}_S2Hand.tif'))
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/LabelHand/{chave}_LabelHand.tif', f'{LB}/{chave}_LabelHand.tif'))
t0 = time.perf_counter()
with cf.ThreadPoolExecutor(16) as pool:
    novos = sum(pool.map(baixar, trabalhos))
open(f'{SP}/flood_train_data.txt', 'w').write('\n'.join(TREINO) + '\n')
open(f'{SP}/flood_valid_data.txt', 'w').write('\n'.join(VAL) + '\n')
print(f'{len(trabalhos)} arquivos conferidos, {novos} baixados agora, em {time.perf_counter() - t0:.0f} s. '
      f'Treino {len(TREINO)}, val {len(VAL)}; listas iguais às do repositório.')

## Treino e inferência

Treino como na grade da semana 3 (`prereg/piloto_lr.md`), com a semente trocada. Depois do treino, a inferência roda em 32 bits sobre os 89 chips inteiros da val, com a mesma escala e normalização do treino, e grava a margem de logit (água menos terra) de cada pixel pontuado, os pesos treináveis e o registro da execução. A mIoU em 32 bits deve ficar perto da `val/mIoU` que o treino registra em 16 bits.

In [ ]:
import gc, warnings
import torch
import lightning as L
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from importlib.metadata import version
try:
    from terratorch.tasks import SemanticSegmentationTask
    from terratorch.datamodules import Sen1Floods11NonGeoDataModule
except ModuleNotFoundError as erro:
    raise RuntimeError('TerraTorch não está instalado nesta sessão. Rode a célula 1 e confira que ela imprime terratorch 1.2.11.') from erro

warnings.filterwarnings('ignore', category=UserWarning)
assert torch.cuda.is_available(), 'Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > T4.'
ACELERADOR, PRECISAO, DISPOSITIVO = 'gpu', '16-mixed', 'cuda'
BANDAS = ['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2']
ESPERADO_M, TOLERANCIA_M = 15.1, 0.1
LOTE, EPOCAS, LR_R1 = 16, 50, 5e-4
SEMENTES = (101, 102)
TRABALHADORES = min(8, os.cpu_count() or 2)

def datamodule():
    para_tensor = lambda: ToTensorV2(transpose_mask=False)
    return Sen1Floods11NonGeoDataModule(
        data_root=DADOS, batch_size=LOTE, num_workers=TRABALHADORES, bands=BANDAS,
        train_transform=[A.RandomCrop(height=224, width=224, p=1.0), A.HorizontalFlip(p=0.5),
                         A.VerticalFlip(p=0.5), para_tensor()],
        val_transform=[para_tensor()], test_transform=[para_tensor()],
        drop_last=True, constant_scale=0.0001, no_data_replace=0.0, no_label_replace=-1, use_metadata=False)

def tarefa_r1(lr):
    args = dict(
        backbone='prithvi_eo_v2_300_tl', backbone_pretrained=True, backbone_ckpt_path=BACKBONE_PT,
        backbone_bands=BANDAS,
        necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]}, {'name': 'ReshapeTokensToImage'}],
        decoder='UperNetDecoder', decoder_channels=256, decoder_scale_modules=True,
        head_dropout=0.1, num_classes=2, rescale=True)
    tarefa = SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory', model_args=args, loss='ce', ignore_index=-1,
        lr=lr, optimizer='AdamW', optimizer_hparams={'weight_decay': 0.05},
        scheduler='CosineAnnealingLR', scheduler_hparams={'T_max': 50, 'eta_min': 0},
        freeze_backbone=True, plot_on_val=0)
    tarefa.configure_models()
    treinaveis = sum(p.numel() for p in tarefa.parameters() if p.requires_grad)
    assert abs(treinaveis / 1e6 - ESPERADO_M) <= TOLERANCIA_M, f'R1: {treinaveis / 1e6:.2f} M treináveis'
    return tarefa

def inferir_val(modelo):
    dm = datamodule()
    dm.setup('validate')
    ds = dm.val_dataset
    chaves = [chave_de(f) for f in ds.image_files]
    assert chaves == [chave_de(f) for f in ds.segmentation_mask_files], 'Imagens e rótulos da val fora de ordem.'
    assert sorted(chaves) == sorted(VAL), 'Chips da val diferentes da lista registrada.'
    carregador = DataLoader(ds, batch_size=8, shuffle=False, num_workers=TRABALHADORES)
    modelo = modelo.to(DISPOSITIVO).float().eval()
    margens, rotulos, tamanhos = [], [], []
    with torch.no_grad(), torch.autocast(device_type=DISPOSITIVO, enabled=False):
        for lote in carregador:
            imagem = dm.aug({'image': lote['image'].to(DISPOSITIVO), 'mask': lote['mask'].to(DISPOSITIVO)})['image']
            assert imagem.dtype == torch.float32 and imagem.ndim == 4 and imagem.shape[1] == len(BANDAS), tuple(imagem.shape)
            z = modelo(imagem).output.float()
            assert tuple(z.shape[-2:]) == tuple(lote['mask'].shape[-2:]), f'Saída {tuple(z.shape)}'
            m = (z[:, 1] - z[:, 0]).cpu().numpy()
            for mi, yi in zip(m, lote['mask'].numpy()):
                pontuado = (yi == 0) | (yi == 1)
                margens.append(mi[pontuado].astype(np.float32))
                rotulos.append(yi[pontuado].astype(np.uint8))
                tamanhos.append(int(pontuado.sum()))
    assert len(tamanhos) == len(chaves)
    return chaves, np.array(tamanhos, dtype=np.int64), np.concatenate(margens), np.concatenate(rotulos)

def miou_binario(pred, y):
    ious = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious))

for semente in SEMENTES:
    registro = os.path.join(RES, f'piloto_R1_s{semente}.json')
    arq_margens = os.path.join(RES, f'margens_R1_s{semente}.npz')
    if os.path.exists(registro) and os.path.exists(arq_margens) and json.load(open(registro)).get('situacao') == 'ok':
        print(f'R1 semente {semente}: já feito ({registro}).')
        continue

    L.seed_everything(semente, workers=True)
    tarefa, dm = tarefa_r1(LR_R1), datamodule()
    trainer = L.Trainer(accelerator=ACELERADOR, devices=1, precision=PRECISAO, max_epochs=EPOCAS,
                        check_val_every_n_epoch=EPOCAS, num_sanity_val_steps=0, logger=False,
                        enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False)
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    trainer.fit(tarefa, datamodule=dm)
    minutos = (time.perf_counter() - t0) / 60
    metricas = {k: float(v) for k, v in trainer.callback_metrics.items() if k.startswith('val/')}
    assert metricas and all(np.isfinite(list(metricas.values()))), f'Métricas não finitas: {metricas}'

    chaves, tamanhos, margem, rotulo = inferir_val(tarefa.model)
    miou_32 = miou_binario((margem > 0).astype(np.uint8), rotulo)
    np.savez_compressed(arq_margens, chaves=np.array(chaves), tamanhos=tamanhos, margem=margem, rotulo=rotulo)
    arq_pesos = os.path.join(RES, f'pesos_treinaveis_R1_s{semente}.pt')
    torch.save({k: v.detach().cpu() for k, v in tarefa.model.state_dict().items() if not k.startswith('encoder.')}, arq_pesos)

    linha = dict(
        regime='R1', semente=semente, lr=LR_R1, epocas=EPOCAS, lote=LOTE,
        gerado_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        situacao='ok', metricas_val_treino=metricas, miou_val_32bits=miou_32,
        minutos=minutos, pico_memoria_gb=torch.cuda.max_memory_allocated() / 1e9,
        pixels_pontuados=int(tamanhos.sum()), chips=len(chaves), chips_com_pixel=int((tamanhos > 0).sum()),
        backbone_sha256=SHA_BACKBONE, gpu=torch.cuda.get_device_name(0),
        versoes={p: version(p) for p in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy')},
        arquivos=dict(margens=arq_margens, pesos=arq_pesos))
    json.dump(linha, open(registro, 'w'), indent=2, ensure_ascii=False)
    print(f"R1 semente {semente}: val/mIoU no treino {metricas.get('val/mIoU', float('nan')):.4f}, "
          f"mIoU em 32 bits {miou_32:.4f}, {minutos:.1f} min, pico {linha['pico_memoria_gb']:.1f} GB, "
          f"{linha['pixels_pontuados']:,} pixels em {linha['chips_com_pixel']} chips")

    del tarefa, trainer, dm, margem, rotulo
    gc.collect()
    torch.cuda.empty_cache()

print('\nGuarde a saída desta célula.')

## Análise

Para cada modelo: taxa de erro, mIoU, AUROC de detecção de erro, τ conforme (α = 0,10, calculado e aplicado na própria val) e taxa de erro não sinalizado. Depois, bootstrap por chip (B = 2.000, mesmo sorteio para todos os modelos): correlação entre modelos e erro-padrão de cada um. Por fim, a correlação dos AUROC por chip e o desvio entre as duas sementes. As conferências da regra aparecem na saída: histograma contra valor exato, mIoU em 16 e 32 bits, pixels por chip iguais aos do R0 e α maior que a taxa de erro.

In [ ]:
from sklearn.metrics import roc_auc_score
from scipy.stats import pearsonr, spearmanr

ALFA, B, SEMENTE_BOOT = 0.10, 2000, 0
LARGURA, TOPO = 0.005, 40.0
NBINS = int(round(TOPO / LARGURA)) + 1
MIN_POR_CLASSE = 50

def caixa(c):
    i = np.floor(np.where(np.isfinite(c), c, TOPO) / LARGURA)
    return np.minimum(i, NBINS - 1).astype(np.int64)

def preparar(nome, chaves, tamanhos, s, erro, confianca, msp, ordem):
    """s: 1 - p(classe verdadeira); confianca: chave de ordenação (maior = mais confiante); ordem: escore do AUROC_e exato."""
    n = len(s)
    nivel = min(1.0, np.ceil((n + 1) * (1 - ALFA)) / n)
    qhat = float(np.quantile(s, nivel, method='higher'))
    tau = 1 - qhat
    sinalizado = msp >= tau
    offs = np.concatenate([[0], np.cumsum(tamanhos)])
    he = np.zeros((len(chaves), NBINS), np.float64); hc = np.zeros_like(he)
    nerr, nok, nao_sinal = (np.zeros(len(chaves), np.int64) for _ in range(3))
    auroc_chip = np.full(len(chaves), np.nan)
    b = caixa(confianca)
    for i in range(len(chaves)):
        sl = slice(offs[i], offs[i + 1])
        e = erro[sl]
        he[i] = np.bincount(b[sl][e], minlength=NBINS); hc[i] = np.bincount(b[sl][~e], minlength=NBINS)
        nerr[i], nok[i] = e.sum(), (~e).sum()
        nao_sinal[i] = (e & sinalizado[sl]).sum()
        if nerr[i] >= MIN_POR_CLASSE and nok[i] >= MIN_POR_CLASSE:
            auroc_chip[i] = roc_auc_score(e, ordem[sl])
    return dict(nome=nome, chaves=list(chaves), tamanhos=np.asarray(tamanhos), qhat=qhat, tau=tau,
                he=he, hc=hc, nerr=nerr, nok=nok, nao_sinal=nao_sinal, auroc_chip=auroc_chip,
                auroc_exato=float(roc_auc_score(erro, ordem)), taxa_erro=float(erro.mean()),
                nao_sinalizado=float((erro & sinalizado).sum() / erro.sum()), n=n)

def auroc_hist(he, hc):
    acima = np.cumsum(hc[..., ::-1], axis=-1)[..., ::-1] - hc
    return (he * (acima + 0.5 * hc)).sum(-1) / (he.sum(-1) * hc.sum(-1))

def miou_binario(pred, y):
    ious = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious))

MODELOS, MIOU, CONTAGEM = {}, {}, {}

# R1: margens salvas pela célula 5
for semente in SEMENTES:
    d = np.load(os.path.join(RES, f'margens_R1_s{semente}.npz'))
    m, y = d['margem'].astype(np.float64), d['rotulo'].astype(np.int64)
    t = m * (2 * y - 1)
    s = 1.0 / (1.0 + np.exp(t))
    erro = (m > 0).astype(np.int64) != y
    msp = 1.0 / (1.0 + np.exp(-np.abs(m)))
    nome = f'R1_s{semente}'
    MODELOS[nome] = preparar(nome, d['chaves'], d['tamanhos'], s, erro, np.abs(m), msp, -np.abs(m))
    MIOU[nome] = miou_binario((m > 0).astype(np.int64), y)
    CONTAGEM[nome] = dict(zip(d['chaves'].tolist(), d['tamanhos'].tolist()))
    del d, m, y, t, s, erro, msp

# R0: escores salvos em 2026-06-07, só os chips da val
escores = np.load(CAL['cal_scores.npy'], mmap_mode='r')
rotulos = np.load(CAL['cal_labels.npy'], mmap_mode='r')
tam = np.load(CAL['cal_chip_sizes.npy']).astype(np.int64)
nomes = [x.decode() if isinstance(x, bytes) else str(x) for x in np.load(CAL['cal_chip_names.npy'], allow_pickle=True)]
assert int(tam.sum()) == len(escores) == len(rotulos) and len(nomes) == len(tam), 'Arquivos do R0 com tamanhos incoerentes.'
offs = np.concatenate([[0], np.cumsum(tam)])
por_chave = {}
for i, nome in enumerate(nomes):
    k = chave_de(nome)
    if k in VAL_SET:
        assert k not in por_chave, f'Chip repetido nos escores do R0: {k}'
        s = np.asarray(escores[offs[i]:offs[i + 1]], dtype=np.float64)
        yy = np.asarray(rotulos[offs[i]:offs[i + 1]])
        ok = np.isin(yy, [0, 1]) & np.isfinite(s)
        por_chave[k] = (s[ok], yy[ok].astype(np.int64))
assert set(por_chave) == VAL_SET, f'Chips da val sem escore do R0: {sorted(VAL_SET - set(por_chave))}'
chaves_r0 = MODELOS[f'R1_s{SEMENTES[0]}']['chaves']
s0 = np.concatenate([por_chave[k][0] for k in chaves_r0])
y0 = np.concatenate([por_chave[k][1] for k in chaves_r0])
tam0 = np.array([len(por_chave[k][0]) for k in chaves_r0])
erro0 = s0 > 0.5
with np.errstate(divide='ignore'):
    conf0 = np.abs(np.log1p(-s0) - np.log(s0))
MODELOS['R0'] = preparar('R0', chaves_r0, tam0, s0, erro0, conf0, np.maximum(s0, 1 - s0), np.minimum(s0, 1 - s0))
MIOU['R0'] = miou_binario(np.where(erro0, 1 - y0, y0), y0)
CONTAGEM['R0'] = dict(zip(chaves_r0, tam0.tolist()))
del s0, y0, erro0, conf0

NOMES = list(MODELOS)
for nome in NOMES:
    assert MODELOS[nome]['chaves'] == chaves_r0, f'{nome}: ordem de chips diferente.'
divergentes = [k for k in chaves_r0 if len({CONTAGEM[n][k] for n in NOMES}) > 1]
print(f'Pixels pontuados por chip iguais nos rótulos atuais e nos do R0: '
      f'{"sim, nos " + str(len(chaves_r0)) + " chips" if not divergentes else "NÃO, em " + str(len(divergentes)) + " chips: " + str(divergentes[:10])}')

com_pixel = np.array([all(MODELOS[n]['tamanhos'][i] > 0 for n in NOMES) for i in range(len(chaves_r0))])
idx = np.where(com_pixel)[0]
K = len(idx)

rng = np.random.default_rng(SEMENTE_BOOT)
sorteios = rng.integers(0, K, size=(B, K))
W = np.zeros((B, K))
for r in range(B):
    W[r] = np.bincount(sorteios[r], minlength=K)
BOOT_AUROC, BOOT_NS = {}, {}
for nome in NOMES:
    M = MODELOS[nome]
    he, hc = M['he'][idx], M['hc'][idx]
    a = np.empty(B)
    for ini in range(0, B, 250):
        fim = min(B, ini + 250)
        a[ini:fim] = auroc_hist(W[ini:fim] @ he, W[ini:fim] @ hc)
    BOOT_AUROC[nome] = a
    BOOT_NS[nome] = (W @ M['nao_sinal'][idx]) / (W @ M['nerr'][idx])
    M['auroc_hist'] = float(auroc_hist(he.sum(0), hc.sum(0)))
media = 'R1_media'
BOOT_AUROC[media] = np.mean([BOOT_AUROC[f'R1_s{s}'] for s in SEMENTES], axis=0)
BOOT_NS[media] = np.mean([BOOT_NS[f'R1_s{s}'] for s in SEMENTES], axis=0)

def correlacao(x, y):
    if x.std() == 0 or y.std() == 0:
        return None
    return float(np.corrcoef(x, y)[0, 1])

PARES = [(f'R1_s{SEMENTES[0]}', 'R0'), (f'R1_s{SEMENTES[1]}', 'R0'), (media, 'R0'), (f'R1_s{SEMENTES[0]}', f'R1_s{SEMENTES[1]}')]
CORR = {f'{a} x {b}': dict(auroc_e=correlacao(BOOT_AUROC[a], BOOT_AUROC[b]),
                           nao_sinalizado=correlacao(BOOT_NS[a], BOOT_NS[b]))
        for a, b in PARES}
formatar = lambda v: 'indefinida' if v is None else f'{v:.3f}'

validos = np.all([np.isfinite(MODELOS[n]['auroc_chip']) for n in NOMES], axis=0)
CORR_CHIP = {}
for a, b in PARES:
    if a == media:
        continue
    xa, xb = MODELOS[a]['auroc_chip'][validos], MODELOS[b]['auroc_chip'][validos]
    if validos.sum() < 3:
        CORR_CHIP[f'{a} x {b}'] = dict(chips=int(validos.sum()), pearson=None, spearman=None)
        continue
    CORR_CHIP[f'{a} x {b}'] = dict(chips=int(validos.sum()), pearson=float(pearsonr(xa, xb)[0]), spearman=float(spearmanr(xa, xb)[0]))

def dp_sementes(valores):
    return float(abs(valores[0] - valores[1]) / np.sqrt(2))
r1 = [f'R1_s{s}' for s in SEMENTES]
DP = dict(auroc_e=dp_sementes([MODELOS[n]['auroc_exato'] for n in r1]),
          nao_sinalizado=dp_sementes([MODELOS[n]['nao_sinalizado'] for n in r1]),
          miou=dp_sementes([MIOU[n] for n in r1]))

REG = {s: json.load(open(os.path.join(RES, f'piloto_R1_s{s}.json'))) for s in SEMENTES}
print(f'\nVal: {K} chips com pixel pontuado; bootstrap por chip, B = {B}, gerador {SEMENTE_BOOT}.\n')
print(f"{'modelo':10s} {'erro':>7s} {'mIoU':>7s} {'AUROC_e':>8s} {'EP boot':>8s} {'hist-exato':>11s} {'tau':>8s} {'não sinal.':>10s} {'alfa>erro':>9s}")
for nome in NOMES:
    M = MODELOS[nome]
    print(f"{nome:10s} {M['taxa_erro']:7.4f} {MIOU[nome]:7.4f} {M['auroc_exato']:8.4f} {BOOT_AUROC[nome].std(ddof=1):8.4f} "
          f"{M['auroc_hist'] - M['auroc_exato']:+11.6f} {M['tau']:8.4f} {M['nao_sinalizado']:10.4f} {str(ALFA > M['taxa_erro']):>9s}")
for s in SEMENTES:
    print(f"R1_s{s}: val/mIoU do treino (16 bits) {REG[s]['metricas_val_treino'].get('val/mIoU', float('nan')):.4f} "
          f"contra {REG[s]['miou_val_32bits']:.4f} em 32 bits")
print('\nCorrelação entre modelos, réplicas do bootstrap (primária):')
for par, v in CORR.items():
    print(f"  {par:22s} AUROC_e {formatar(v['auroc_e'])} | não sinalizado {formatar(v['nao_sinalizado'])}")
print(f'\nCorrelação dos AUROC_e por chip (secundária, chips com pelo menos {MIN_POR_CLASSE} erros e {MIN_POR_CLASSE} acertos em todos):')
for par, v in CORR_CHIP.items():
    if v['pearson'] is None:
        print(f"  {par:22s} {v['chips']} chips: poucos para calcular")
    else:
        print(f"  {par:22s} {v['chips']} chips | Pearson {v['pearson']:.3f} | Spearman {v['spearman']:.3f}")
print(f"\nDesvio entre sementes do R1 (|x1 - x2| / raiz de 2, 1 grau de liberdade): AUROC_e {DP['auroc_e']:.4f} | "
      f"não sinalizado {DP['nao_sinalizado']:.4f} | mIoU {DP['miou']:.4f}")

SAIDA = dict(
    gerado_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    regra='prereg/piloto_sementes.md', alfa=ALFA, B=B, gerador_bootstrap=SEMENTE_BOOT, largura_caixa=LARGURA, topo_caixa=TOPO,
    min_por_classe_chip=MIN_POR_CLASSE, chips_val=len(chaves_r0), chips_com_pixel=K,
    chips_divergentes_r0=divergentes, arquivos_r0=CAL,
    modelos={n: dict(taxa_erro=MODELOS[n]['taxa_erro'], miou=MIOU[n], auroc_e=MODELOS[n]['auroc_exato'],
                     auroc_e_hist=MODELOS[n]['auroc_hist'], ep_bootstrap_auroc_e=float(BOOT_AUROC[n].std(ddof=1)),
                     qhat=MODELOS[n]['qhat'], tau=MODELOS[n]['tau'], nao_sinalizado=MODELOS[n]['nao_sinalizado'],
                     ep_bootstrap_nao_sinalizado=float(BOOT_NS[n].std(ddof=1)), pixels=MODELOS[n]['n'],
                     precondicao_alfa_maior_que_erro=bool(ALFA > MODELOS[n]['taxa_erro']))
             for n in NOMES},
    registro_treino={f'R1_s{s}': REG[s] for s in SEMENTES},
    correlacao_bootstrap=CORR, correlacao_por_chip=CORR_CHIP, dp_entre_sementes=DP,
    por_chip={n: dict(chaves=MODELOS[n]['chaves'], pixels=MODELOS[n]['tamanhos'].tolist(), erros=MODELOS[n]['nerr'].tolist(),
                      nao_sinalizados=MODELOS[n]['nao_sinal'].tolist(),
                      auroc_e=[None if not np.isfinite(x) else float(x) for x in MODELOS[n]['auroc_chip']])
              for n in NOMES})
caminho = os.path.join(RES, 'piloto_sementes.json')
json.dump(SAIDA, open(caminho, 'w'), indent=2, ensure_ascii=False)
print(f'\nGravado em {caminho}')
print('Guarde a saída desta célula inteira.')